In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib

In [2]:
df = pd.read_csv('../data/BVBRC_genome_amr.csv', index_col=0)
df

In [3]:
df.info()

In [4]:
# Check the distribution of the target variable
df['Resistant Phenotype'].value_counts()

In [5]:
# Check antibiotic distribution and count unique antibiotics
df['Antibiotic'].value_counts()

In [21]:
# plot antibiotic distribution
plt.figure(figsize=(18,10))
sns.countplot(data=df, x='Antibiotic', order=df['Antibiotic'].value_counts().index)
plt.xticks(rotation=90)
plt.xlabel('Antibiotic')
plt.ylabel('Count')
plt.title('Distribution of Antibiotics')
plt.show()

In [6]:
len(df.Antibiotic.unique())

In [7]:
len(df['Genome Name'].unique())

## Merge Data

Merge Phonotypic and Genome data together on Genome ID and Genome Name such that it keeps corresponding entries from each dataset

In [8]:
# getting genome data
genome_data = pd.read_csv('../data/BVBRC_genome (1).csv', index_col=0)


# right join to combine both dataframes on the genome id
combined_df = pd.merge(genome_data, df, on=['Genome ID','Genome Name'], how='right')
combined_df

In [9]:
len(combined_df['Genome Name'].unique())

## AMR Matrix Transformation
We will now transform the data so that:
1. Each **Genome** is a row.
2. Each **Antibiotic** is a column.
3. The values are encoded: **Susceptible (0), Intermediate (1), Resistant (2), Nonsusceptible(2)**.


In [10]:
# Map Phenotypes to Numbers
phenotype_map = {
    'Susceptible': 0,
    'Intermediate': 1,
    'Resistant': 2,
    'Nonsusceptible': 2, # Treat NonSusceptible as Resistant for simplicity
}

# Filter out rows where 'Resistant Phenotype' is missing or not in our map
clean_df = combined_df[combined_df['Resistant Phenotype'].isin(phenotype_map.keys())].copy()
clean_df['Phenotype_Code'] = clean_df['Resistant Phenotype'].map(phenotype_map)
print("Unique antibiotics after filtering:", len(clean_df.Antibiotic.unique()))
print("Unique genomes after filtering:", len(clean_df['Genome Name'].unique()))

# Pivot the table indexing with Genome Name
amr_matrix = clean_df.pivot_table(
    index= ['Genome Name'], 
    columns='Antibiotic', 
    values='Phenotype_Code',
    aggfunc='max' # If there are duplicates, take the higher resistance (worst-case scenario)
)

amr_matrix



## Cleaning Antibiotics

Removing antibiotics which have more missing values or just a few actual Phenotypic label



In [11]:
# checking for missing values
missing_values = amr_matrix.isnull().sum().sort_values()
print("Missing values per antibiotic:\n", missing_values, )

In [ ]:
# count plot of missing values per antibiotics
missing_df = missing_values.sort_values(ascending=False).reset_index()
missing_df.columns = ['Antibiotic', 'Missing_Count']

plt.figure(figsize=(22, 9))
sns.barplot(data=missing_df, x='Antibiotic', y='Missing_Count', color='steelblue')
plt.xticks(rotation=90)
plt.title('Missing Values per Antibiotic')
plt.xlabel('Antibiotic')
plt.ylabel('Number of Missing Values')
plt.tight_layout()
plt.show()




In [13]:
# drop antibiotics with more than 50% missing values
threshold = len(amr_matrix) * 0.5
amr_matrix = amr_matrix.loc[:, ~(amr_matrix.isnull().sum() > threshold)]
amr_matrix


In [14]:
# Get genome ID of the 7884 genomes in amr_matrix as a list
genome_id_map = combined_df.drop_duplicates(subset=['Genome Name']).set_index('Genome Name')['Genome ID']
target_genome_ids = amr_matrix.index.map(genome_id_map).tolist()

print(f"Number of Genome IDs retrieved: {len(target_genome_ids)}")
target_genome_ids

# save list to file
with open("../data/genome_ids.txt", "w") as f:
    for genome_id in target_genome_ids:
        f.write(f"{genome_id}\n")
        


In [15]:
# add genome ID to amr_matrix
amr_matrix.insert(0, 'Genome ID', target_genome_ids )

# reset index to have genome name as a column
amr_matrix.reset_index(inplace=True)
amr_matrix

In [17]:
amr_matrix.isnull().sum().sort_values()

In [23]:
# see antibiotic distribution
amr_matrix.info()

In [24]:
# checking class distribution for each antibiotics
amr_matrix.ampicillin.value_counts()

In [36]:
amr_matrix[amr_matrix['Genome ID']==562.2287]

In [ ]:
# plot distribution of classes for all antibiotics
import math

antibiotics = amr_matrix.drop(columns=['Genome Name', 'Genome ID'])
num_antibiotics = len(antibiotics.columns)

# Determine grid size (e.g., 5 columns, and as many rows as needed)
cols = 5
rows = math.ceil(num_antibiotics / cols)

# Create a large figure with subplots
fig, axes = plt.subplots(rows, cols, figsize=(20, 3 * rows))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

# Define colors for the classes: Susceptible (0), Intermediate (1), Resistant (2)
colors = {0.0: '#2ca02c', 1.0: '#ff7f0e', 2.0: '#d62728'} 

for i, col in enumerate(antibiotics.columns):
    # Get value counts and sort the index so 0, 1, 2 are in order
    counts = antibiotics[col].value_counts().sort_index()
    
    # Map the index to the corresponding colors
    bar_colors = [colors.get(x, 'gray') for x in counts.index]
    
    axes[i].bar(counts.index.astype(str), counts.values, color=bar_colors)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Count')

# Hide any extra empty subplots if the grid is larger than the number of antibiotics
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Create a custom legend for the whole figure
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ca02c', label='Susceptible (0)'),
    Patch(facecolor='#ff7f0e', label='Intermediate (1)'),
    Patch(facecolor='#d62728', label='Resistant (2)')
]
fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.98]) 
plt.show()

In [13]:
# read data
kmer = pd.read_csv('../kmer-data.csv')
kmer.head()

In [15]:
kmer.describe()

In [ ]:
kmers_long = pd.read_csv("../kmer-data-long.csv")  # GenomeID, kmer, count

kmer_matrix = kmers_long.pivot_table(
    index="GenomeID",
    columns="kmer",
    values="count",
    aggfunc="sum",
    fill_value=0,
).reset_index()  # <- makes GenomeID a column

kmer_matrix.head()
# Optionally make it sparse for memory
# kmer_matrix.iloc[:, 1:] = kmer_matrix.iloc[:, 1:].astype("Sparse[int]")

In [4]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             from google.colab import files
import os

# Upload the file
print("Select the kmer-data-long.csv file to upload:")
uploaded = files.upload()

# Verify the file was uploaded
for filename in uploaded.keys():
    print(f"Uploaded: {filename}")

In [1]:
import pandas as pd

dp = pd.read_csv('/content/kmer-data-long.csv')

dp.head()